# Lab4.5 — Avaliação perceptiva de imagens por pares (v6 corrigida)

**Drive único • Grupos produtor e avaliador independentes • Comparação cega**

Este notebook permite que **G1 avalie G2** ou **G2 avalie G1**, sem alterar o significado das variáveis. Os arquivos de todos os grupos ficam sob `MyDrive/Lab4_CSM/`.

**Ordem:** (0) configuração → (1) preparar pacote do produtor, **apenas quando necessário** → (2) avaliar pacote do produtor → (3) analisar notas recebidas. A etapa 1 **não é executada automaticamente**.

**Escala:** 0 = nenhuma diferença percebida; 1 = pequena; 2 = moderada; 3 = grande. Cada par compara a imagem original com uma versão JPEG, com posição A/B aleatória. O gabarito fica na pasta privada do produtor.

## 0. Configuração — defina quem produz e quem avalia

In [ ]:
from pathlib import Path
import json, random
import pandas as pd
from PIL import Image
from IPython.display import display
import ipywidgets as widgets
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

RAIZ = Path('/content/drive/MyDrive/Lab4_CSM')
GRUPO_PRODUTOR = 'G4'     # Dono das imagens; exemplo: G2
GRUPO_AVALIADOR = 'G3'    # Quem atribui as notas; exemplo: G1
AVALIADOR = 'A1'          # A1, A2, A3 ou A4
NOME_IMAGEM = 'G4_1792112691.png'  # Nome da imagem ORIGINAL do produtor, para a etapa 1

PARES = {'G1':'G2','G2':'G1','G3':'G4','G4':'G3','G5':'G6','G6':'G5'}
assert PARES.get(GRUPO_AVALIADOR) == GRUPO_PRODUTOR, 'Os grupos devem ser parceiros (G1/G2, G3/G4 ou G5/G6).'
assert AVALIADOR in ('A1','A2','A3','A4'), 'AVALIADOR deve ser A1, A2, A3 ou A4.'

PASTA_PRODUTOR = RAIZ / GRUPO_PRODUTOR
PASTA_AVALIADOR = RAIZ / GRUPO_AVALIADOR
ORIG = PASTA_PRODUTOR / 'imagens' / 'originais'
JPEG = PASTA_PRODUTOR / 'imagens' / 'jpeg'
PACOTE = PASTA_PRODUTOR / 'pacote_imagem'
DESTINO = PASTA_AVALIADOR / 'avaliacoes_imagem'
RESULTADOS_RECEBIDOS = PASTA_PRODUTOR / 'imagens' / 'resultados_recebidos'
GABARITO = PASTA_PRODUTOR / 'imagens' / 'gabarito_imagem_PRIVADO.csv'

print(f'PRODUTOR: {GRUPO_PRODUTOR} | Original: {ORIG / NOME_IMAGEM}')
print(f'AVALIADOR: {GRUPO_AVALIADOR} ({AVALIADOR})')
print('Leitura dos pares:', PACOTE)
print('Gravação das notas:', DESTINO)


## 1. Produtor — preparar os pares (opcional)

Execute **somente quando precisar criar ou atualizar o pacote**. O arquivo original deve existir em `GRUPO_PRODUTOR/imagens/originais/` e as três versões JPEG produzidas no Lab4.3 em `GRUPO_PRODUTOR/imagens/jpeg/`, com sufixos `_q2.jpg`, `_q20.jpg`, `_q31.jpg`.

Ao atualizar o pacote, a etapa 2 deve ser executada novamente para atualizar as imagens mostradas na tela. A preparação substitui os pares existentes do produtor.

In [ ]:
def preparar_imagens():
    original = ORIG / NOME_IMAGEM
    assert original.is_file(), f'Original não encontrado em {GRUPO_PRODUTOR}: {original}'
    entradas = [(q, JPEG / f'{original.stem}_q{q}.jpg') for q in (2, 20, 31)]
    faltando = [str(arq) for _, arq in entradas if not arq.is_file()]
    assert not faltando, 'JPEGs ausentes (execute Lab4.3):\n' + '\n'.join(faltando)
    PACOTE.mkdir(parents=True, exist_ok=True)
    ordem = entradas.copy()
    random.SystemRandom().shuffle(ordem)
    chave = []
    with Image.open(original) as im:
        referencia = im.convert('RGB').copy()
    for i, (q, arquivo) in enumerate(ordem, 1):
        with Image.open(arquivo) as im:
            teste = im.convert('RGB').copy()
        assert referencia.size == teste.size, f'Dimensões diferentes: {arquivo}'
        pid = f'P{i:02d}'
        lado_original = random.SystemRandom().choice(('A', 'B'))
        a, b = (referencia, teste) if lado_original == 'A' else (teste, referencia)
        a.save(PACOTE / f'{pid}_A.png')
        b.save(PACOTE / f'{pid}_B.png')
        chave.append({'par': pid, 'q_ffmpeg': q, 'original_lado': lado_original})
    GABARITO.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(chave).to_csv(GABARITO, index=False)
    (PACOTE / 'info.json').write_text(json.dumps({
        'produtor': GRUPO_PRODUTOR, 'avaliador': GRUPO_AVALIADOR,
        'imagem_original': original.name
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Pacote atualizado:', PACOTE)
    print('Imagem usada:', original)
    print('Gabarito privado:', GABARITO)
    print('ATENÇÃO: reexecute a etapa 2 para recarregar as imagens.')

# Para criar/atualizar, remova o # da linha seguinte e execute esta célula:
# preparar_imagens()


## 2. Avaliador — observar e pontuar os três pares

O grupo **avaliador** abre os pares do **produtor**, sem consultar o gabarito. Execute esta célula novamente depois de qualquer atualização do pacote. Os arquivos são lidos novamente do Drive a cada execução.

In [ ]:
assert PACOTE.is_dir(), f'Pacote inexistente: {PACOTE}. Prepare o pacote na etapa 1.'
info_arquivo = PACOTE / 'info.json'
assert info_arquivo.is_file(), f'Falta info.json em {PACOTE}'
info = json.loads(info_arquivo.read_text(encoding='utf-8'))
assert info.get('produtor') == GRUPO_PRODUTOR, f'Produtor inesperado: {info.get("produtor")}'
print(f'Você é {GRUPO_AVALIADOR} e avalia imagens de {GRUPO_PRODUTOR}.')
print('Imagem declarada:', info.get('imagem_original', '(não informada)'))
print('Pasta de leitura:', PACOTE)

avaliacoes = []
for i in range(1, 4):
    pid = f'P{i:02d}'
    a, b = PACOTE / f'{pid}_A.png', PACOTE / f'{pid}_B.png'
    assert a.is_file() and b.is_file(), f'Par incompleto: {a} / {b}'
    print(f'{pid} — A versus B (0 a 3)')
    display(widgets.HBox([
        widgets.VBox([widgets.Label('Imagem A'), widgets.Image(value=a.read_bytes(), format='png', width=360)]),
        widgets.VBox([widgets.Label('Imagem B'), widgets.Image(value=b.read_bytes(), format='png', width=360)])
    ]))
    escala = widgets.IntSlider(value=0, min=0, max=3, step=1, description=pid, continuous_update=False)
    display(escala)
    avaliacoes.append((pid, escala))

botao = widgets.Button(description='Salvar avaliação', button_style='success')
saida = widgets.Output()
display(botao, saida)

def salvar_avaliacao(_):
    with saida:
        saida.clear_output()
        dados = pd.DataFrame([{
            'grupo_avaliador': GRUPO_AVALIADOR,
            'grupo_produtor': GRUPO_PRODUTOR,
            'avaliador': AVALIADOR,
            'imagem_original': info.get('imagem_original', ''),
            'par': pid, 'diferenca_0_3': escala.value
        } for pid, escala in avaliacoes])
        DESTINO.mkdir(parents=True, exist_ok=True)
        RESULTADOS_RECEBIDOS.mkdir(parents=True, exist_ok=True)
        arquivo_avaliador = DESTINO / f'{GRUPO_PRODUTOR}_{AVALIADOR}.csv'
        arquivo_produtor = RESULTADOS_RECEBIDOS / f'{GRUPO_AVALIADOR}_{AVALIADOR}.csv'
        dados.to_csv(arquivo_avaliador, index=False)
        dados.to_csv(arquivo_produtor, index=False)
        print('Notas gravadas em:', arquivo_avaliador)
        print('Cópia entregue ao produtor:', arquivo_produtor)

botao.on_click(salvar_avaliacao)


## 3. Produtor — interpretar as avaliações recebidas

Esta seção usa os resultados de **GRUPO_PRODUTOR** e seu gabarito privado. Não apresente o gabarito aos avaliadores antes do encerramento.

In [ ]:
if GABARITO.is_file() and RESULTADOS_RECEBIDOS.is_dir():
    arquivos = sorted(RESULTADOS_RECEBIDOS.glob('*.csv'))
    if arquivos:
        notas = pd.concat([pd.read_csv(p) for p in arquivos], ignore_index=True)
        if 'imagem_original' in notas.columns:
            notas = notas[notas['imagem_original'].ne('') & notas['imagem_original'].eq(NOME_IMAGEM)]
        if notas.empty:
            print('Nenhuma avaliação recebida para a imagem configurada:', NOME_IMAGEM)
        else:
            resultado = notas.merge(pd.read_csv(GABARITO), on='par', how='left')
            display(resultado.groupby('q_ffmpeg')['diferenca_0_3'].agg(['count','mean','std']))
    else:
        print('Ainda não há avaliações recebidas.')
else:
    print('Prepare o pacote e aguarde avaliações.')
